In [2]:
import random

def Diff(selected_snps, T):
    """
    计算 selected_snps 与 T 的差异。
    这里给一个示例：假设 T 是目标 SNP 集合，
    Diff = 不匹配的 SNP 数量。
    """
    return sum(1 for s in selected_snps if s not in T)


def RandomizedHaplotypeSearch(S, T, k):
    # Step 1: 随机选择 k 个 SNP 作为初始解
    bestSNPs = random.sample(S, k)

    while True:
        currentSNPs = bestSNPs.copy()
        improved = False

        # Step 2: 尝试替换 currentSNPs 中的每个 SNP
        for s in currentSNPs:
            for s_prime in (set(S) - set(currentSNPs)):
                # 替换 s → s'
                S_prime = currentSNPs.copy()
                S_prime.remove(s)
                S_prime.append(s_prime)

                # Step 3: 如果更好，则更新 bestSNPs
                if Diff(S_prime, T) < Diff(bestSNPs, T):
                    bestSNPs = S_prime
                    improved = True

        # Step 4: 如果没有改进，则返回
        if not improved:
            return bestSNPs


In [3]:
S = ["A", "B", "C", "D", "E", "F"]
T = ["A", "C", "E"]  # 假设目标 haplotype
k = 3

result = RandomizedHaplotypeSearch(S, T, k)
print("Best SNPs:", result)


Best SNPs: ['A', 'E', 'C']


In [4]:
import random
import math

def Diff(selected_snps, T):
    """
    示例差异函数：计算 selected_snps 与 T 的不匹配数量。
    可根据实际任务替换。
    """
    return sum(1 for s in selected_snps if s not in T)


def SimulatedAnnealingHaplotypeSearch(S, T, k,
                                      T_init=1.0,
                                      T_min=1e-4,
                                      alpha=0.95,
                                      max_iter=500):
    # 随机初始化
    current = random.sample(S, k)
    current_score = Diff(current, T)
    best = current[:]
    best_score = current_score

    T_now = T_init

    while T_now > T_min:
        for _ in range(max_iter):
            # 随机选择一个 SNP 替换
            s = random.choice(current)
            s_prime = random.choice(list(set(S) - set(current)))

            candidate = current[:]
            candidate.remove(s)
            candidate.append(s_prime)

            candidate_score = Diff(candidate, T)

            # ΔE = 新解 - 旧解
            delta = candidate_score - current_score

            # 若更好，接受；若更差，以概率接受
            if delta < 0 or random.random() < math.exp(-delta / T_now):
                current = candidate
                current_score = candidate_score

                # 更新全局最优
                if current_score < best_score:
                    best = current[:]
                    best_score = current_score

        # 降温
        T_now *= alpha

    return best


In [5]:
from collections import defaultdict

class Node:
    def __init__(self, name=None):
        self.name = name          # 可选：节点名字（如 'root' 或 leaf id）
        self.children = []        # 子节点列表
        self.characters = set()   # 在该节点“引入”的 SNP 列索引集合（突变发生处）
        self.leaves = []          # 挂在该节点下的个体（行索引）

    def add_child(self, child):
        self.children.append(child)


def compute_O_sets(A):
    """
    计算每一列 j 的 O_j = { 所有在第 j 列为 1 的行索引 }
    返回: 列数 n, 以及 O_sets: List[set]
    """
    m = len(A)
    n = len(A[0])
    O_sets = []
    for j in range(n):
        Oj = {i for i in range(m) if A[i][j] == 1}
        O_sets.append(Oj)
    return n, O_sets


def check_pairwise_compatibility(O_sets):
    """
    检查所有列是否两两兼容：
    对任意 i, j:
      Oi ⊆ Oj 或 Oj ⊆ Oi 或 Oi ∩ Oj = ∅
    若不兼容，返回 False
    """
    n = len(O_sets)
    for i in range(n):
        for j in range(i + 1, n):
            Oi, Oj = O_sets[i], O_sets[j]
            inter = Oi & Oj
            if inter:
                # 有交集，则必须有包含关系
                if not (Oi <= Oj or Oj <= Oi):
                    return False
    return True


def build_character_tree(O_sets):
    """
    根据 O_sets 构造“字符树”（SNP 层级树）。
    思路：
      - 按 |Oj| 从大到小排序（出现次数多的在上层）
      - 对每个 Oj，找到一个最小的“真超集”作为父节点；若无，则挂在 root 下
    返回: root 节点, 以及每个列 j 对应的节点
    """
    n = len(O_sets)
    indices = list(range(n))
    # 按集合大小从大到小排序
    indices.sort(key=lambda j: len(O_sets[j]), reverse=True)

    root = Node(name="root")
    char_node = {}  # j -> Node

    for j in indices:
        Oj = O_sets[j]
        parent = root
        parent_set = None

        # 在已放置的列中找一个最小真超集作为父节点
        for k, node_k in char_node.items():
            Ok = O_sets[k]
            if Oj < Ok:  # 真子集
                if parent_set is None or Ok < parent_set:
                    parent = node_k
                    parent_set = Ok

        # 在 parent 下创建该字符节点（可能多个字符共享一个节点，这里每列一个节点）
        node = Node(name=f"char_{j}")
        node.characters.add(j)
        parent.add_child(node)
        char_node[j] = node

    return root, char_node


def attach_leaves(A, root, char_node):
    """
    将每一行（个体）挂到树上：
      对于个体 i：
        找到它拥有的所有 SNP 列 j
        这些列对应的节点在字符树中形成一条从根向下的路径（因为列兼容）
        我们把该个体挂在这条路径最深的节点下
    """
    m = len(A)
    n = len(A[0])

    # 为了快速找“最深节点”，我们给每个节点一个深度
    def assign_depths(node, depth=0):
        node.depth = depth
        for child in node.children:
            assign_depths(child, depth + 1)

    assign_depths(root)

    for i in range(m):
        # 该个体拥有的 SNP 列
        cols = [j for j in range(n) if A[i][j] == 1]
        if not cols:
            # 没有任何突变，挂在 root
            root.leaves.append(i)
            continue

        # 找到这些列对应的节点中“最深”的那个
        candidate_nodes = [char_node[j] for j in cols]
        deepest = max(candidate_nodes, key=lambda node: node.depth)
        deepest.leaves.append(i)


def PerfectPhylogeny(A):
    """
    主函数：给定 SNP 矩阵 A（0/1），构建 perfect phylogeny。
    返回 root 节点。
    若矩阵不满足 pairwise compatibility，则抛出异常。
    """
    n, O_sets = compute_O_sets(A)

    if not check_pairwise_compatibility(O_sets):
        raise ValueError("Matrix is not phylogenetic: columns are not pairwise compatible.")

    root, char_node = build_character_tree(O_sets)
    attach_leaves(A, root, char_node)
    return root


In [6]:
def print_tree(node, indent=0):
    prefix = "  " * indent
    chars = ",".join(str(c) for c in sorted(node.characters)) if node.characters else "-"
    leaves = ",".join(str(i) for i in node.leaves) if node.leaves else "-"
    print(f"{prefix}Node(name={node.name}, chars={chars}, leaves={leaves})")
    for child in node.children:
        print_tree(child, indent + 1)


In [8]:
A = [
    [1,0,0,0],
    [1,1,0,0],
    [1,1,1,0],
    [1,1,1,1],
]

root = PerfectPhylogeny(A)
print_tree(root)

Node(name=root, chars=-, leaves=-)
  Node(name=char_0, chars=0, leaves=0)
    Node(name=char_1, chars=1, leaves=1)
      Node(name=char_2, chars=2, leaves=2)
        Node(name=char_3, chars=3, leaves=3)


In [12]:
def mtDNA_survival_children(n):
    """
    返回：第 n 代“孩子”中，仍然存在某条原始 mtDNA 谱系的概率
    n >= 1；n=1 时按教材 Note：结果为 1.0
    """
    if n == 1:
        return 1.0  # 第一代孩子一定都有原始 mtDNA
    
    q = 0.0  # q_0 = 0：第 0 代（原始女性）没有灭绝
    # 递推到 q_{n-1}
    for _ in range(n - 1):
        q = 0.25 + 0.5 * q + 0.25 * (q ** 2)
    return 1 - q


In [15]:
for n in range(1, 8):
    print(n, round(mtDNA_survival_children(n), 4))



1 1.0
2 0.75
3 0.6094
4 0.5165
5 0.4498
6 0.3992
7 0.3594
